# Hindi Dataset Creation:

- Created the dataset by downloading the audio files from youtube and tehn extracting the audio features using the librosa lib.

In [ ]:
!pip install yt-dlp pydub librosa numpy soundfile

In [ ]:
import os
import librosa
import numpy as np
import yt_dlp
import pandas as pd
from pydub import AudioSegment

# Folder to store audio files
AUDIO_FOLDER = "audio_files"
os.makedirs(AUDIO_FOLDER, exist_ok=True)

# Input CSV file containing song data
csv_file = "/kaggle/input/hindi-songs/combined_file.csv"
df = pd.read_csv(csv_file)

# Output CSV file for extracted features (created in the working directory)
output_csv = "audio_features.csv"

# Feature weights from the image
FEATURE_WEIGHTS = {
    "tempo": 4.5,
    "valence": 4.0,
    "popularity": 4.0,
    "danceability": 3.5,
    "energy": 4.5,
    "loudness": 3.0,
    "speechiness": 2.5,
    "acousticness": 3.5,
    "instrumentalness": 3.5,
    "liveness": 3.0,
    "explicit": 2.5
}

# Check if output CSV already exists
if os.path.exists(output_csv):
    processed_df = pd.read_csv(output_csv)
    processed_songs = set(processed_df["Song Title"].tolist())  # Avoid re-processing songs
else:
    processed_songs = set()

def download_audio(song_title):
    """Download the audio file using YouTube search."""
    search_query = f"{song_title} Hindi song audio"
    output_template = os.path.join(AUDIO_FOLDER, f"{song_title}.%(ext)s")
    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": output_template,
        "noplaylist": True,
        "quiet": True,
        "postprocessors": [
            {"key": "FFmpegExtractAudio", "preferredcodec": "mp3", "preferredquality": "192"}
        ],
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            ydl.download([f"ytsearch:{search_query}"])
            print(f"Downloaded: {song_title}")
            return output_template.replace("%(ext)s", "mp3")  # Return actual file path
        except Exception as e:
            print(f"Failed to download {song_title}: {e}")
            return None

def extract_audio_features(file_path):
    """Extract audio features using Librosa."""
    try:
        # Load the audio file
        y, sr = librosa.load(file_path, sr=None)
        
        # Feature extraction
        features = {}
        
        # Duration
        duration = librosa.get_duration(y=y, sr=sr)
        features["duration_ms"] = duration * 1000
        
        # Tempo (Weight: 4.5)
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        features["tempo"] = tempo
        
        # Energy (Weight: 4.5)
        # Energy is the sum of squares of the signal values, normalized by the frame length
        features["energy"] = np.mean(librosa.feature.rms(y=y))
        
        # Loudness (Weight: 3.0)
        # Approximate loudness as the average decibel value
        S = np.abs(librosa.stft(y))
        features["loudness"] = np.mean(librosa.amplitude_to_db(S, ref=np.max))
        
        # Danceability (Weight: 3.5)
        # Approximate danceability with beat strength and regularity
        _, beats = librosa.beat.beat_track(y=y, sr=sr)
        if len(beats) > 0:
            beat_strength = np.mean([y[i] ** 2 for i in beats if i < len(y)])
            beat_intervals = np.diff(beats)
            regularity = 1.0 / (np.std(beat_intervals) + 1e-10)
            features["danceability"] = np.mean([beat_strength, regularity])
        else:
            features["danceability"] = 0.0
        
        # Speechiness (Weight: 2.5)
        # Higher values for more speech-like signals
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        features["speechiness"] = np.mean(np.std(mfccs, axis=1))
        
        # Acousticness (Weight: 3.5)
        # Higher values for more acoustic signals
        spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        # Acoustic tracks often have lower spectral centroids
        features["acousticness"] = 1.0 - (np.mean(spectral_centroid) / (sr/2))
        
        # Instrumentalness (Weight: 3.5)
        # Higher values for more instrumental signals
        # Use spectral flatness as a proxy for instrumentalness
        spectral_flatness = librosa.feature.spectral_flatness(y=y)[0]
        features["instrumentalness"] = np.mean(spectral_flatness)
        
        # Liveness (Weight: 3.0)
        # Detect audience sounds and applause
        # Higher values for more live sounds
        onset_env = librosa.onset.onset_strength(y=y, sr=sr)
        features["liveness"] = np.mean(onset_env)
        
        # Valence (Weight: 4.0)
        # Approximating musical positiveness through spectral content
        # Higher values for more positive/happy sounds
        spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        features["valence"] = np.mean(spectral_contrast)
        
        # Explicit content (Weight: 2.5)
        # Hard to detect without proper lyrics analysis
        # We'll use a placeholder value of 0 (not explicit)
        features["explicit"] = 0.0
        
        # Popularity (Weight: 4.0)
        # We'll use a placeholder value of 50 (neutral popularity)
        # In a real application, this would be fetched from a streaming service API
        features["popularity"] = 50.0
        
        # Additional features from the original code
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        features["chroma"] = np.mean(chroma)
        
        spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        features["spectral_contrast"] = np.mean(spectral_contrast)
        
        zero_crossings = librosa.zero_crossings(y, pad=False)
        features["zero_crossings"] = np.mean(zero_crossings)
        
        return features
        
    except Exception as e:
        print(f"Error extracting metadata for {file_path}: {e}")
        return None

# Feature normalization removed as requested

def save_to_csv(song_title, features):
    """Save extracted features to the new CSV file."""
    data = {"Song Title": song_title}
    data.update(features)
    new_df = pd.DataFrame([data])
    
    # Append to CSV if it exists, otherwise create a new file
    if os.path.exists(output_csv):
        new_df.to_csv(output_csv, mode="a", header=False, index=False)
    else:
        new_df.to_csv(output_csv, mode="w", header=True, index=False)
    
    print(f"Updated CSV for {song_title}")

# Process songs one by one
for index, row in df.iloc[0:500].iterrows():  # Processing songs from row 1000 to 2000
    song_title = row["Song Title"]
    if pd.notna(song_title) and song_title not in processed_songs:  # Ensure valid and new song
        file_path = os.path.join(AUDIO_FOLDER, f"{song_title}.mp3")
        if not os.path.exists(file_path):  # Skip if already downloaded
            file_path = download_audio(song_title)
        
        if file_path and os.path.exists(file_path):  # Extract features if download was successful
            features = extract_audio_features(file_path)
            
            if features:
                save_to_csv(song_title, features)  # Save to new CSV

print("Processing complete. All features extracted and saved in 'audio_features.csv'.")

# Artist Names and Genre Extraction using the Spotify API

In [2]:
!pip install spotipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.5/261.5 kB 7.0 MB/s eta 0:00:00:00:01


In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import pandas as pd

# Replace with your own Spotify API credentials
CLIENT_ID = "fc0e125e455946c09ffcee398b070fba"
CLIENT_SECRET = "298722aa95d34c0a8342b79cadccc483"

# Authenticate with Spotify API
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(client_id=CLIENT_ID, client_secret=CLIENT_SECRET))

def get_genre(song_name):
    query = f"track:{song_name}"
    results = sp.search(q=query, type="track", limit=1)
    
    if results["tracks"]["items"]:
        track = results["tracks"]["items"][0]
        artist = track["artists"][0]
        artist_name = artist["name"]
        artist_id = artist["id"]
        artist_info = sp.artist(artist_id)
        genres = artist_info["genres"]  # Returns a list of genres
        return artist_name, genres
    
    return None, None

# Load song names from CSV file
df = pd.read_csv("/kaggle/input/hindi-songs/clean_data.csv")  # Replace with the actual filename

# Assuming the CSV has a column named 'song_name'
song_names = df["Song Title"].dropna().tolist()  # Drop NaNs and convert to list

# Process first 100 songs
for i, song in enumerate(song_names[:100]):
    artist, genre = get_genre(song)
    print(f"{i+1}. Song: {song} | Artist: {artist} | Genre: {genre}")


# Sentiment Analysis

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
import json
import torch

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("SamLowe/roberta-base-go_emotions")
model = AutoModelForSequenceClassification.from_pretrained("SamLowe/roberta-base-go_emotions")

# Load JSON data (Hindi songs dataset)
with open("/kaggle/input/genre-prediction-dataset/rhyme_analysis.json", "r", encoding="utf-8") as file:
    data = json.load(file)

df_json = pd.DataFrame(data)

# Load CSV data (contains English translations)
df_csv = pd.read_csv("/kaggle/input/hindi-songs/clean_data.csv")  # Ensure it has 'Song Title', 'Hindi Lyrics', 'English Translations'

# Ensure consistency in string matching
def clean_text(text):
    return str(text).strip().lower() if isinstance(text, str) else ""

# Add English translations to JSON dataset
translations = {}

for _, row in df_csv.iterrows():
    key = (clean_text(row["Song Title"]), clean_text(row["Hindi Lyrics"]))
    translations[key] = row["English Translations"]

# Merge English translations into JSON dataset
df_json["English Translations"] = df_json.apply(
    lambda row: translations.get((clean_text(row["Song Title"]), clean_text(row["Hindi Lyrics"])), ""), axis=1
)

# Initialize list to store sentiment predictions
top_emotions_list = []

# Process each song
for i, entry in df_json.iterrows():
    title = entry["Song Title"]
    text = entry["English Translations"]  # Get matched English lyrics

    # Skip if lyrics are empty
    if not isinstance(text, str) or len(text.strip()) == 0:
        top_emotions_list.append("")
        continue

    # Truncate text for efficient processing
    truncated_text = text[:500]

    # Tokenize input text
    inputs = tokenizer(truncated_text, return_tensors="pt")
    outputs = model(**inputs)
    logits = outputs.logits

    # Get top 4 sentiment predictions
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    sorted_indices = torch.argsort(probabilities, descending=True)

    # Convert indices to labels
    top_emotions = []
    for idx in sorted_indices[0]:
        emotion = model.config.id2label[idx.item()]
        if emotion != "neutral":  # Exclude "Neutral"
            top_emotions.append(emotion)
        if len(top_emotions) == 4:  # Limit to top 4
            break

    # Convert to comma-separated string
    emotions_str = ", ".join(top_emotions)
    top_emotions_list.append(emotions_str)

    # Print progress
    print(f"[{i+1}/{len(df_json)}] Processed '{title}' → Emotions: {emotions_str}")

# Add sentiments to DataFrame
df_json["Sentiment"] = top_emotions_list

# Save updated JSON file
df_json.to_json("hindi_songs_with_sentiment.json", orient="records", indent=4, force_ascii=False)

print("✅ Sentiment analysis completed! File saved as 'hindi_songs_with_sentiment.json'.")


# Rhyme Analysis:

In [ ]:
!pip install transformers torch sentencepiece


In [ ]:
import re
import pandas as pd
from collections import defaultdict
import ast
import string


def extract_rhyme_scheme(lyrics_list):
    """
    Extract rhyme scheme from a list of lyrics lines

    Args:
        lyrics_list: List of strings containing lyrics lines

    Returns:
        Dictionary with stanza-wise rhyme schemes
    """
    # Clean up the list - remove empty lines and non-lyric elements
    clean_lyrics = []
    for line in lyrics_list:
        if isinstance(line, str) and line.strip() and not line.strip().isdigit() and line.strip() != 'Lyrics':
            clean_lyrics.append(line.strip())

    # Split into stanzas (separated by empty lines in the original)
    stanzas = []
    current_stanza = []

    for line in clean_lyrics:
        if line:
            current_stanza.append(line)
        else:
            if current_stanza:
                stanzas.append(current_stanza)
                current_stanza = []

    # Add the last stanza if it's not empty
    if current_stanza:
        stanzas.append(current_stanza)

    # Extract the rhyme scheme for each stanza
    results = {}
    overall_rhyme_pattern = []
    overall_rhyme_dict = {}
    next_rhyme = 0

    for stanza_idx, stanza in enumerate(stanzas):
        stanza_pattern = []

        # Get the last word/sound of each line
        line_endings = []
        for line in stanza:
            # For Hindi in English script, we focus on the last few characters
            # This is a simplified approach - could be refined with phonetic analysis
            ending = line.split()[-1].lower()
            # Take the last few characters as a proxy for sound (adjust based on results)
            sound_ending = ending[-3:] if len(ending) >= 3 else ending
            line_endings.append(sound_ending)

        # Map similar sounds to the same letter
        for ending in line_endings:
            if ending not in overall_rhyme_dict:
                overall_rhyme_dict[ending] = string.ascii_uppercase[next_rhyme]
                next_rhyme = (next_rhyme + 1) % 26

            stanza_pattern.append(overall_rhyme_dict[ending])

        results[f"Stanza_{stanza_idx+1}"] = {
            "lines": stanza,
            "pattern": stanza_pattern
        }
        overall_rhyme_pattern.extend(stanza_pattern)

    # Add the overall pattern
    results["overall_pattern"] = overall_rhyme_pattern
    results["rhyme_dict"] = {v: k for k, v in overall_rhyme_dict.items()}

    return results


def process_lyrics_dataset(file_path, lyrics_column="Hindi Lyrics"):
    """
    Process a dataset of lyrics and extract rhyme schemes

    Args:
        file_path: Path to CSV file with lyrics data
        lyrics_column: Name of the column containing lyrics (default: "Hindi Lyrics")

    Returns:
        Original DataFrame with added columns for rhyme schemes
    """
    # Load the dataset
    df = pd.read_csv(file_path)

    # Check if the lyrics column exists
    if lyrics_column not in df.columns:
        available_columns = ', '.join(df.columns)
        raise ValueError(
            f"Column '{lyrics_column}' not found in dataset. Available columns: {available_columns}")

    # Add new columns for rhyme analysis
    df['rhyme_scheme'] = None
    df['formatted_scheme'] = None
    df['rhyme_dict'] = None

    # Process each row
    for idx, row in df.iterrows():
        try:
            # Get the lyrics - handle both string representations of lists and actual lists
            lyrics_data = row[lyrics_column]
            if isinstance(lyrics_data, str):
                try:
                    # Try to parse as a Python list
                    lyrics_list = ast.literal_eval(lyrics_data)
                except:
                    # If parsing fails, split by newline
                    lyrics_list = lyrics_data.split('\n')
            else:
                lyrics_list = lyrics_data

            # Skip empty lyrics
            if not lyrics_list or (isinstance(lyrics_list, list) and len(lyrics_list) <= 1):
                continue

            # Extract rhyme scheme
            rhyme_data = extract_rhyme_scheme(lyrics_list)

            # Create a formatted representation
            formatted_scheme = ""
            for stanza_name, stanza_data in rhyme_data.items():
                if stanza_name.startswith("Stanza"):
                    formatted_scheme += f"\n{stanza_name}:\n"
                    for i, (line, pattern) in enumerate(zip(stanza_data["lines"], stanza_data["pattern"])):
                        formatted_scheme += f"{line} ({pattern})\n"

            # Update the dataframe
            df.at[idx, 'rhyme_scheme'] = str(rhyme_data["overall_pattern"])
            df.at[idx, 'formatted_scheme'] = formatted_scheme.strip()
            df.at[idx, 'rhyme_dict'] = str(rhyme_data["rhyme_dict"])

            # Print progress
            if idx % 100 == 0:
                print(f"Processed {idx} entries...")

        except Exception as e:
            print(f"Error processing row {idx}: {str(e)}")

    return df


# Example usage
if __name__ == "__main__":
    # For a single lyrics example
    # example_lyrics = ['Lyrics', 'Yeh dosti hum nahin todhenge', 'Todhenge dum magar', 'Tera saath na chhodenge', 'Yeh dosti hum nahin todhenge', 'Todhenge dum magar', 'Tera saath na chhodenge', '', 'Arre meri jeet teri jeet', 'Teri haar meri haar', 'Sunn aye mere yaar', 'Tera gham mera gham', 'Meri jaan teri jaan', 'Aaisa apna pyar', 'Jaan pe bhi khelenge', 'Tere liye le lenge', 'Jaan pe bhi khelenge', 'Tere liye le lenge', 'Sabse dushmani', '', 'Yeh dosti hum nahin todhenge', 'Todhenge dum magar', 'Tera saath na chhodenge', '', 'Logon ko aate hain do nazar', 'Hum magar dekho do nahin', 'Arre ho judaa ya khafa', 'Aye khuda hain dua aaisa ho nahin', 'Khaana peena saath hain', 'Marna jeena saath hain', 'Khaana peena saath hain', 'Marna jeena saath hain', 'Saari zindagi', '', 'Yeh dosti hum nahin todhenge', 'Todhenge dum magar', 'Tera saath na chhodenge', 'Yeh dosti hum nahin todhenge', 'Todhenge dum magar', 'Tera saath na chhodenge', '174']

    # result = extract_rhyme_scheme(example_lyrics)
    # print("Single Lyrics Analysis:")
    # for stanza_name, stanza_data in result.items():
    #     if stanza_name.startswith("Stanza"):
    #         print(f"\n{stanza_name}:")
    #         for i, (line, pattern) in enumerate(zip(stanza_data["lines"], stanza_data["pattern"])):
    #             print(f"{line} ({pattern})")

    # For processing your specific dataset
    dataset_path = 'merged_songs_renumbered.csv'  # Update this path

    # Process the dataset with Hindi lyrics
    df_with_rhyme_schemes = process_lyrics_dataset(dataset_path, lyrics_column="Hindi Lyrics")

    # Save the results
    output_path = 'final_audio_features_with_rhyme_schemes.csv'
    df_with_rhyme_schemes.to_json("rhyme_analysis.json", orient="records", indent=4)

    # df_with_rhyme_schemes.to_csv(output_path, index=False)
    print(f"Processed dataset saved to {output_path}")


# English Songs Dataset creation

- the audio file downloading from youtube and the audio feature extraction from librosa module

In [ ]:
!pip install yt_dlp

In [ ]:
import os
import librosa
import numpy as np
import yt_dlp
import pandas as pd
import gc
import time

# ---------------------------
# Configuration & Constants
# ---------------------------

AUDIO_FOLDER = "audio_files"
os.makedirs(AUDIO_FOLDER, exist_ok=True)

csv_file = "/kaggle/input/hindi-songs/combined_file.csv"
df = pd.read_csv(csv_file, low_memory=True)  # Load CSV efficiently

output_csv = "audio_features.csv"

# Check if output CSV exists
if os.path.exists(output_csv):
    processed_df = pd.read_csv(output_csv, usecols=["Song Title"])
    processed_songs = set(processed_df["Song Title"].dropna().tolist())
else:
    processed_songs = set()

# ---------------------------
# Functions
# ---------------------------

def download_audio_soundcloud(song_title):
    """Download audio from SoundCloud using yt_dlp."""
    search_query = f"scsearch:{song_title} official"
    output_template = os.path.join(AUDIO_FOLDER, f"{song_title}.%(ext)s")
    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": output_template,
        "noplaylist": True,
        "quiet": True,
        "postprocessors": [
            {"key": "FFmpegExtractAudio", "preferredcodec": "mp3", "preferredquality": "192"}
        ],
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([search_query])
        return output_template.replace("%(ext)s", "mp3")
    except Exception as e:
        print(f"⚠️ Failed to download {song_title}: {e}")
        return None

def extract_audio_features(file_path):
    """Extract audio features using Librosa with memory optimization."""
    try:
        y, sr = librosa.load(file_path, sr=None, duration=30)  # Load only 30s

        features = {
            "duration_ms": librosa.get_duration(y=y, sr=sr) * 1000,
            "tempo": librosa.beat.tempo(y=y, sr=sr)[0],
            "energy": np.mean(librosa.feature.rms(y=y)),
            "loudness": np.mean(librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)),
            "speechiness": np.mean(np.std(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)),
            "acousticness": 1.0 - (np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)) / (sr/2)),
            "instrumentalness": np.mean(librosa.feature.spectral_flatness(y=y)),
            "liveness": np.mean(librosa.onset.onset_strength(y=y, sr=sr)),
            "valence": np.mean(librosa.feature.spectral_contrast(y=y, sr=sr)),
            "chroma": np.mean(librosa.feature.chroma_stft(y=y, sr=sr)),
            "explicit": 0.0,  # Placeholder
            "popularity": 50.0  # Placeholder
        }

        del y, sr  # Free up memory
        gc.collect()

        return features
    except Exception as e:
        print(f"⚠️ Error extracting features for {file_path}: {e}")
        return None

def save_to_csv(song_title, features):
    """Append extracted features to the CSV file."""
    new_df = pd.DataFrame([{**{"Song Title": song_title}, **features}])
    new_df.to_csv(output_csv, mode="a", header=not os.path.exists(output_csv), index=False)
    print(f"✅ Saved features for {song_title}")

# ---------------------------
# Main Processing (Batch-wise)
# ---------------------------

df_batch = df.iloc[1600:]  # Process songs 301 to 600
batch_size = 50

for i in range(0, len(df_batch), batch_size):
    df_sub_batch = df_batch.iloc[i : i + batch_size]

    for index, row in df_sub_batch.iterrows():
        song_title = row["Song Title"]
        if pd.notna(song_title) and song_title not in processed_songs:
            print(f"🔍 Processing: {song_title}")

            file_path = os.path.join(AUDIO_FOLDER, f"{song_title}.mp3")
            if not os.path.exists(file_path):
                file_path = download_audio_soundcloud(song_title)

            if file_path and os.path.exists(file_path):
                features = extract_audio_features(file_path)
                if features:
                    save_to_csv(song_title, features)
                else:
                    print(f"⚠️ Could not extract features for {song_title}")
            else:
                print(f"⚠️ File not available for {song_title}")

            time.sleep(1)  # Avoid API rate limit issues

    gc.collect()  # Free memory after every batch

print("✅ Processing complete. All features saved to 'audio_features.csv'.")


# Other code cells:

- we used more helping codes like script for the merging of the dataset as we had to run the cod emultiple times in parallel so as to pace up the process.
- and other files that were just helping and did not make a major impact on the dataset.